In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, global_add_pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import itertools
import random
import json
import numpy as np

print('PyTorch:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

raw = load_jsonl('../data/cayley_dict_unique_100.jsonl')
raw += load_jsonl('../data/cayley_dict_unique_51_200.jsonl')[791:]
raw += load_jsonl('../data/cayley_ten_gens_each_50.jsonl')
raw += load_jsonl('../data/cayley_dict_five_gens_each_51_150.jsonl')

print(f'Loaded {len(raw)} graphs')
print('Keys:', list(raw[0].keys()))

In [ ]:
def build_relation_dataset(cayley_dicts):
    """
    Build PyG Data objects for the Relation Space architecture.

    Node features x:
        Shape [n, 1].  x[0] = 1 (identity), x[v] = 0 for v > 0.
        This is the delta initialisation anchored at the identity.

    Edge attribute edge_gen:
        Shape [num_edges].  Integer index of which generator produced each edge.
        Used to select the right A_s matrix during message passing.

    max_gens is returned so all models use the same number of A_s matrices.
    """
    max_gens = max(d['num_generators'] for d in cayley_dicts)
    dataset = []
    for d in cayley_dicts:
        n = d['num_vertices']
        edge_index = torch.tensor(d['edge_index'], dtype=torch.long).t().contiguous()  # [2, E]

        # Map edge labels → 0-indexed generator integers
        unique_labels = d['edge_label'][:d['num_generators']]
        label_to_idx = {lbl: i for i, lbl in enumerate(unique_labels)}
        edge_gen = torch.tensor([label_to_idx[lbl] for lbl in d['edge_label']], dtype=torch.long)

        # Delta initialisation at identity (vertex 0)
        x = torch.zeros(n, 1)
        x[0, 0] = 1.0

        y = torch.tensor([d['is_abelian']], dtype=torch.long)
        dataset.append(Data(x=x, edge_index=edge_index, edge_gen=edge_gen,
                            num_generators=d['num_generators'], y=y))
    return dataset, max_gens


dataset, max_gens = build_relation_dataset(raw)
print(f'max generators across all graphs: {max_gens}')
print(f'abelian: {sum(int(d.y) for d in dataset)}  '
      f'non-abelian: {sum(1 - int(d.y) for d in dataset)}')

In [ ]:
def make_balanced(dataset, seed=1):
    ab  = [d for d in dataset if int(d.y) == 1]
    nab = [d for d in dataset if int(d.y) == 0]
    rng = random.Random(seed)
    balanced = ab + rng.sample(nab, len(ab))
    rng.shuffle(balanced)
    return balanced

def stratified_split(dataset, seed=42):
    labels = [int(d.y) for d in dataset]
    idx = list(range(len(dataset)))
    tr, tmp = train_test_split(idx, test_size=0.30, random_state=seed, stratify=labels)
    tmp_labels = [labels[i] for i in tmp]
    va, te = train_test_split(tmp, test_size=0.50, random_state=seed, stratify=tmp_labels)
    return [dataset[i] for i in tr], [dataset[i] for i in va], [dataset[i] for i in te]

balanced = make_balanced(dataset)
train_set, val_set, test_set = stratified_split(balanced)
print(f'Train {len(train_set)}  Val {len(val_set)}  Test {len(test_set)}')

In [ ]:
from torch_geometric.nn import MessagePassing

class CayleyRelationConv(MessagePassing):
    """
    h^{l+1}(w) = sigma( sum_s  A_s  h^l(w*s) )

    flow='target_to_source': source node w receives from target w*s.
    Edges are stored as (src=w, tgt=w*s), so x_j = h[tgt] = h[w*s]. Correct.
    """
    def __init__(self, in_channels, out_channels, num_generators, bias=True):
        super().__init__(aggr='add', flow='target_to_source')
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_generators = num_generators
        # One weight matrix per generator
        self.A = nn.ModuleList([
            nn.Linear(in_channels, out_channels, bias=False)
            for _ in range(num_generators)
        ])
        self.bias = nn.Parameter(torch.zeros(out_channels)) if bias else None

    def forward(self, x, edge_index, edge_gen):
        """
        x         : [N, in_channels]
        edge_index: [2, E]  (row 0 = w, row 1 = w*s)
        edge_gen  : [E]     integer in [0, num_generators)
        """
        return self.propagate(edge_index, x=x, edge_gen=edge_gen)

    def message(self, x_j, edge_gen):
        # x_j: [E, in_channels]  (features at w*s, since flow=target_to_source and j=target)
        out = torch.zeros(x_j.size(0), self.out_channels, device=x_j.device)
        for s in range(self.num_generators):
            mask = (edge_gen == s)
            if mask.any():
                out[mask] = self.A[s](x_j[mask])
        return out

    def update(self, aggr_out):
        if self.bias is not None:
            aggr_out = aggr_out + self.bias
        return F.relu(aggr_out)

In [ ]:
class CayleyRelationNet(nn.Module):
    """
    Stacks CayleyRelationConv layers, then pools to a graph-level vector.

    pooling='mean'     : standard mean over all nodes
    pooling='identity' : use only h^L(e) — the identity node embedding
    """
    def __init__(self, num_layers=4, hidden_dim=64, num_generators=7,
                 num_classes=2, pooling='mean'):
        super().__init__()
        self.pooling = pooling
        self.convs = nn.ModuleList()
        for i in range(num_layers):
            in_dim = 1 if i == 0 else hidden_dim
            self.convs.append(CayleyRelationConv(in_dim, hidden_dim, num_generators))
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, edge_index, edge_gen, batch, identity_mask=None):
        for conv in self.convs:
            x = conv(x, edge_index, edge_gen)

        if self.pooling == 'identity' and identity_mask is not None:
            # Read off h^L(e) for each graph: identity_mask marks vertex-0 per graph
            graph_emb = x[identity_mask]
        else:
            graph_emb = global_mean_pool(x, batch)    # [num_graphs, hidden_dim]

        return self.classifier(graph_emb)


# Quick sanity check
sample = dataset[0]
model_test = CayleyRelationNet(num_layers=3, hidden_dim=32, num_generators=max_gens)
out = model_test(
    sample.x,
    sample.edge_index,
    sample.edge_gen,
    torch.zeros(sample.x.size(0), dtype=torch.long),
)
print('Output shape (single graph):', out.shape)  # [1, 2]

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        batch = batch.to(device)
        # Identity mask: first node of each graph in the batch
        identity_mask = _identity_mask(batch)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.edge_gen, batch.batch, identity_mask)
        loss = F.cross_entropy(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    for batch in loader:
        batch = batch.to(device)
        identity_mask = _identity_mask(batch)
        out = model(batch.x, batch.edge_index, batch.edge_gen, batch.batch, identity_mask)
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
        total += batch.y.size(0)
    return correct / total

@torch.no_grad()
def get_all_preds(model, loader, device):
    model.eval()
    all_labels, all_preds = [], []
    for batch in loader:
        batch = batch.to(device)
        identity_mask = _identity_mask(batch)
        out = model(batch.x, batch.edge_index, batch.edge_gen, batch.batch, identity_mask)
        all_preds.extend(out.argmax(dim=1).cpu().tolist())
        all_labels.extend(batch.y.cpu().tolist())
    return all_labels, all_preds

def _identity_mask(batch):
    """Boolean mask picking out vertex 0 (identity) from each graph in the batch."""
    # ptr[i] is the start index of graph i in the batched node tensor
    return batch.ptr[:-1]

In [ ]:
def plot_results(model, history, label='', class_names=['Non-Abelian', 'Abelian']):
    epochs     = history['epochs']
    labels, preds = get_all_preds(model, history['test_loader'], history['device'])
    cm = confusion_matrix(labels, preds)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    pfx = f'[{label}] ' if label else ''

    axes[0].plot(epochs, history['train_loss'], color='steelblue')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{pfx}Training Loss'); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history['val_acc'], color='salmon')
    axes[1].set_ylim(0, 1); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Acc')
    axes[1].axhline(0.5, color='grey', linestyle='--', linewidth=0.8)
    axes[1].set_title(f'{pfx}Validation Accuracy'); axes[1].grid(alpha=0.3)

    im = axes[2].imshow(cm, cmap='Blues')
    fig.colorbar(im, ax=axes[2])
    axes[2].set_title(f'{pfx}Confusion Matrix')
    axes[2].set_xticks(range(2)); axes[2].set_xticklabels(class_names)
    axes[2].set_yticks(range(2)); axes[2].set_yticklabels(class_names)
    axes[2].set_ylabel('True'); axes[2].set_xlabel('Predicted')
    thresh = cm.max() / 2
    for i, j in itertools.product(range(2), range(2)):
        axes[2].text(j, i, cm[i, j], ha='center', va='center',
                     color='white' if cm[i, j] > thresh else 'black')
    plt.tight_layout(); plt.show()
    print(classification_report(labels, preds, target_names=class_names))

In [ ]:
def run_experiment(num_layers=4, hidden_dim=64, pooling='mean',
                   epochs=40, lr=1e-3, seed=42, batch_size=32):
    torch.manual_seed(seed)
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    tr_loader  = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set,   batch_size=batch_size)
    te_loader  = DataLoader(test_set,  batch_size=batch_size)

    model = CayleyRelationNet(
        num_layers=num_layers, hidden_dim=hidden_dim,
        num_generators=max_gens, pooling=pooling
    ).to(dev)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    print(f'layers={num_layers}  hidden={hidden_dim}  pooling={pooling}')
    print(f'params: {sum(p.numel() for p in model.parameters()):,}')

    history = {'epochs': list(range(1, epochs + 1)), 'train_loss': [], 'val_acc': [],
               'test_loader': te_loader, 'device': dev}

    for epoch in range(1, epochs + 1):
        loss    = train_one_epoch(model, tr_loader, optimizer, dev)
        val_acc = evaluate(model, val_loader, dev)
        history['train_loss'].append(loss)
        history['val_acc'].append(val_acc)
        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d} | Loss {loss:.3f} | Val Acc {val_acc:.3f}')

    test_acc = evaluate(model, te_loader, dev)
    print(f'  → Test Accuracy: {test_acc:.4f}')
    return model, history

In [ ]:
# Experiment 1: mean pooling, 4 layers
model1, history1 = run_experiment(num_layers=4, hidden_dim=64, pooling='mean', epochs=40)
plot_results(model1, history1, label='RelationNet-mean')

In [ ]:
# Experiment 2: identity-only pooling — read off h^L(e)
model2, history2 = run_experiment(num_layers=4, hidden_dim=64, pooling='identity', epochs=40)
plot_results(model2, history2, label='RelationNet-identity')

In [ ]:
# Experiment 3: deeper network
model3, history3 = run_experiment(num_layers=6, hidden_dim=128, pooling='mean', epochs=40)
plot_results(model3, history3, label='RelationNet-deep')